# EEG Emotion Recognition - Notebook 02: Preprocessing & Zero-Leakage Pipeline

## 1. Signal Processing Rationale
- **Baseline Handling:** In DEAP, each trial starts with 3 seconds (384 samples) of pre-trial fixation baseline followed by 60 seconds of video stimulus. We subtract the channel-wise mean of the baseline to eliminate slow DC drift and isolate stimulus-induced affective variations.
- **Butterworth Bandpass Filter (4–45 Hz):** Removes low-frequency ocular/head movement drift (<4 Hz) and high-frequency noise / AC powerline hum (>45 Hz). Retains the four critical EEG rhythmic bands:
  - **Theta (4–8 Hz):** Emotional arousal, cognitive control, memory retrieval.
  - **Alpha (8–13 Hz):** Mental relaxation, frontal asymmetry (crucial biomarker for Valence!).
  - **Beta (14–30 Hz):** Active concentration, stress, alertness.
  - **Gamma (30–45 Hz):** Higher cognitive processing, cross-modal emotional integration.
- **Temporal Windowing:** A 60-second trial is segmented into 2.0-second sliding windows (256 samples at 128 Hz) with 1.0-second step (50% overlap), yielding 59 windows per trial.
- **Zero Data Leakage:** The `ZeroLeakageStandardizer` fits exclusively on the training participant cohort (s01–s22). Validation (s23–s27) and test (s28–s32) cohorts are transformed using only training statistics.

In [ ]:
# Setup environment and paths
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/DEAP/data_preprocessed_python'
    OUTPUT_DIR = '/content/drive/MyDrive/DEAP/processed'
else:
    DATA_DIR = '../sample_data' if os.path.exists('../sample_data') else './sample_data'
    OUTPUT_DIR = '../data/processed' if os.path.exists('../notebooks') else './data/processed'

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Target Data Directory:   {DATA_DIR}")
print(f"Processed Output Folder: {OUTPUT_DIR}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from src.config import CONFIG
from src.data_loader import DEAPDataLoader
from src.preprocessing import EEGPreprocessor, ZeroLeakageStandardizer
from src.windowing import EEGWindowSegmenter, build_split_datasets

loader = DEAPDataLoader()
preprocessor = EEGPreprocessor()
segmenter = EEGWindowSegmenter()

In [ ]:
# Demonstrate Frequency Filtering (Raw vs 4-45 Hz Bandpass Filter)
sample_files = list(Path(DATA_DIR).glob('s*.dat'))
if sample_files:
    eeg_raw, labels, _ = loader.load_single_participant(str(sample_files[0]))
    trial_0 = eeg_raw[0]  # (32, 8064)
    
    # Baseline subtraction and filtering
    active_eeg = preprocessor.remove_baseline(trial_0)
    filtered_eeg = preprocessor.bandpass_filter(active_eeg)
    
    # Plot Power Spectral Density (PSD) using Welch's method
    fs = 128
    f_raw, psd_raw = signal.welch(active_eeg[0], fs=fs, nperseg=256)
    f_filt, psd_filt = signal.welch(filtered_eeg[0], fs=fs, nperseg=256)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6))
    ax1.plot(f_raw, 10*np.log10(psd_raw + 1e-8), label="Raw Signal PSD", color='red', alpha=0.7)
    ax1.plot(f_filt, 10*np.log10(psd_filt + 1e-8), label="Filtered Signal (4-45 Hz)", color='blue', lw=1.5)
    ax1.axvspan(4, 45, color='green', alpha=0.1, label="Target Band (Theta-Alpha-Beta-Gamma)")
    ax1.set_title("Power Spectral Density (PSD) Comparison (Welch's Method)", fontweight='bold')
    ax1.set_xlabel("Frequency (Hz)")
    ax1.set_ylabel("Power (dB/Hz)")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Time domain comparison
    t_axis = np.linspace(0, 4.0, 4 * fs, endpoint=False)
    ax2.plot(t_axis, active_eeg[0, :len(t_axis)], label="Unfiltered (active seg)", color='gray', alpha=0.8)
    ax2.plot(t_axis, filtered_eeg[0, :len(t_axis)], label="Bandpass Filtered (4-45 Hz)", color='navy')
    ax2.set_title("First 4 Seconds Waveform Comparison", fontweight='bold')
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Amplitude (µV)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Execute Full Split Building Pipeline
# Fits scaler exclusively on Train cohort, transforms Val and Test
split_datasets = build_split_datasets(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    config=CONFIG,
    save_to_disk=True
)

X_train, y_train = split_datasets['train']
X_val, y_val = split_datasets['val']
X_test, y_test = split_datasets['test']

print("\nDataset Pipeline Summary:")
print(f"- Train Windows: X={X_train.shape}, y={y_train.shape}")
print(f"- Val Windows:   X={X_val.shape}, y={y_val.shape}")
print(f"- Test Windows:  X={X_test.shape}, y={y_test.shape}")
print(f"- Window Shape:  {CONFIG.preprocessing.window_samples} timesteps x {CONFIG.dataset.eeg_channels_count} channels")